# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [49]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [50]:
df = df.assign(revenue=df['qty'] * df['price'])
print(df['revenue'].sum())
print(df.shape)
print("8520 is the total revenue, which was given by multiplying the price and quantity of each order, then summing.\nUsing df.shape prints the # of rows which is 400 as expected and the number of variables")

8520.0
(400, 5)
8520 is the total revenue, which was given by multiplying the price and quantity of each order, then summing.
Using df.shape prints the # of rows which is 400 as expected and the number of variables


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [51]:
print(df.groupby('category')['revenue'].sum().sort_values(ascending=False))
print(df.groupby('category')['revenue'].sum() / df['revenue'].sum() * 100)
by_category = (df.groupby('category')['revenue'].sum().sum())
print("Here we grouped each order by category, then summed the revenue to find the total revenue of each category. \nTo find revenue share as a percent we divided by the total revenue of all orders and mutiplied by 100 to convert to percentage.")

category
Food        4293.0
Merch       1771.5
Drink       1554.0
RainGear     901.5
Name: revenue, dtype: float64
category
Drink       18.239437
Food        50.387324
Merch       20.792254
RainGear    10.580986
Name: revenue, dtype: float64
Here we grouped each order by category, then summed the revenue to find the total revenue of each category. 
To find revenue share as a percent we divided by the total revenue of all orders and mutiplied by 100 to convert to percentage.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [52]:
print(df.groupby('vendor_id')['revenue'].mean().sort_values(ascending=False))
print(df.groupby('vendor_id')['revenue'].count().sort_values(ascending=False))
print("here we grouped by vendor and found the mean of revenue, then we also reported the number of orders for each vendor. We also counted the total number of roders from each vendor to see how many high value orders they have brought in. ")

vendor_id
V-01    22.595745
V-18    21.750000
V-05    20.580645
V-10    20.314286
Name: revenue, dtype: float64
vendor_id
V-18    108
V-10    105
V-01     94
V-05     93
Name: revenue, dtype: int64
here we grouped by vendor and found the mean of revenue, then we also reported the number of orders for each vendor. We also counted the total number of roders from each vendor to see how many high value orders they have brought in. 


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [53]:
print((round(df.groupby('category')['revenue'].sum() / df['revenue'].sum() * 100, 1)))
print("Through grouping and summing we found that 20.8% of revenue is accounted to the merch category ")

category
Drink       18.2
Food        50.4
Merch       20.8
RainGear    10.6
Name: revenue, dtype: float64
Through grouping and summing we found that 20.8% of revenue is accounted to the merch category 


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [54]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = pd.merge(df, vendor_names, on='vendor_id', how='left', validate='many_to_one')
if joined['vendor_name'].isna().any():
    joined['vendor_name'] = joined['vendor_name'].fillna("Unnamed Vendor")
    print("Unmatched vendor located")
else:
    print("No unmatched vendors")

print(joined)

Unmatched vendor located
    vendor_id  category  qty  price  revenue      vendor_name
0        V-10     Drink    2   24.0     48.0  Cav Merch North
1        V-18  RainGear    1   12.0     12.0   Unnamed Vendor
2        V-18     Drink    3    4.5     13.5   Unnamed Vendor
3        V-10      Food    2   12.0     24.0  Cav Merch North
4        V-18     Drink    3    7.5     22.5   Unnamed Vendor
..        ...       ...  ...    ...      ...              ...
395      V-18     Merch    1   12.0     12.0   Unnamed Vendor
396      V-01     Merch    2   24.0     48.0     Hoos Burgers
397      V-10      Food    3    7.5     22.5  Cav Merch North
398      V-18     Merch    2   24.0     48.0   Unnamed Vendor
399      V-10     Drink    1    7.5      7.5  Cav Merch North

[400 rows x 6 columns]


**The unmatched vendor, and what I did about it:** The unmatched vendor is v_18. because the unmatched vendor appears in the data frame as NA. I replaced all NA values with "Unnamed_vendor". Another option would be to add a name for  V_18.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [55]:
pv = pd.pivot_table(joined, values='revenue', index='vendor_name', columns='category', aggfunc='sum', margins=True, margins_name='Total')
print(pv)

category          Drink    Food   Merch  RainGear   Total
vendor_name                                              
Cav Merch North   502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers      171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos     298.5   882.0   489.0     244.5  1914.0
Unnamed Vendor    582.0  1018.5   508.5     240.0  2349.0
Total            1554.0  4293.0  1771.5     901.5  8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [56]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

A. I would tell the vendors to market much more drinks during the games, food and drinks should have a much more similar share of revenue.

B. Q2, revenue by category is the least trustworthy because rain gear and merch can be easily combined or confused between the two, leading to a loss in data quality. A UVA printed umbrella, poncho, or hat is both merch and rain gear.